# BloodBalance — Data Exploration Notebook

**Dataset:** Indian Blood Bank Directory (National Health Portal)

**Purpose:** Exploratory Data Analysis (EDA) and cleaning of the blood-bank directory dataset.

This notebook is designed to run from top to bottom in a fresh Google Colab session.
It replicates the logic from `src/data_loader.py` and `src/analytics.py`.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Run this cell first in Google Colab.
!pip install plotly pandas --quiet

In [ ]:
# ── Cell 2: Mount Google Drive (Colab only) ───────────────────────────────────
# Option A — Google Drive (if you saved the project there)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/bloodbalance/data/raw/blood_banks_raw.csv'

# Option B — Upload directly
# from google.colab import files
# uploaded = files.upload()   # Upload blood_banks_raw.csv when prompted
# DATA_PATH = 'blood_banks_raw.csv'

# Option C — GitHub raw URL (if repo is public)
# DATA_PATH = 'https://raw.githubusercontent.com/YOUR_USERNAME/bloodbalance/main/data/raw/blood_banks_raw.csv'

# ── For local VS Code use: set the path directly ──────────────────────────────
import os
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'raw', 'blood_banks_raw.csv')
print(f'Using data path: {DATA_PATH}')

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import re
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print('Imports OK')

In [ ]:
# ── Cell 4: Load raw data ─────────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH, encoding='latin-1', on_bad_lines='skip')
print(f'Shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}):', list(df_raw.columns))

In [ ]:
# ── Cell 5: Basic profile ─────────────────────────────────────────────────────
print('=== DATA TYPES ===')
print(df_raw.dtypes)
print()
print('=== MISSING VALUES ===')
mv = df_raw.isnull().sum()
mv_pct = (mv / len(df_raw) * 100).round(1)
summary = pd.DataFrame({'Missing': mv, '% Missing': mv_pct})
print(summary[summary['Missing'] > 0].sort_values('% Missing', ascending=False))

In [ ]:
# ── Cell 6: Cleaning pipeline ─────────────────────────────────────────────────
df = df_raw.copy()

# Strip whitespace from column names and string values
df.columns = [c.strip() for c in df.columns]
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

# Rename to snake_case
rename_map = {
    'Sr No': 'sr_no', 'Blood Bank Name': 'bank_name', 'State': 'state',
    'District': 'district', 'City': 'city', 'Address': 'address',
    'Pincode': 'pincode', 'Contact No': 'contact_no', 'Mobile': 'mobile',
    'Email': 'email', 'Category': 'category',
    'Blood Component Available': 'blood_component',
    'Apheresis': 'apheresis', 'Service Time': 'service_time',
    'Latitude': 'latitude', 'Longitude': 'longitude',
}
df.rename(columns=rename_map, inplace=True)

# Normalise service_time
def normalise_service_time(val):
    if pd.isna(val): return 'Unknown'
    s = str(val).strip().lower().replace(' ', '').replace('\\t', '')
    if re.search(r'24', s): return '24x7'
    if re.search(r'\d{1,2}(am|pm|:|h)', s) or 'to' in s or '-' in s: return 'Daytime'
    return 'Other / Unknown'

df['service_time_raw'] = df['service_time'].copy()
df['service_time'] = df['service_time'].apply(normalise_service_time)

# Normalise apheresis
def norm_yna(val):
    if pd.isna(val): return 'Unknown'
    s = str(val).strip().upper()
    return 'Yes' if s == 'YES' else ('No' if s == 'NO' else 'Unknown')

df['apheresis'] = df['apheresis'].apply(norm_yna)
df['blood_component'] = df['blood_component'].apply(norm_yna)
df['category'] = df['category'].fillna('Unknown')

# Coordinate validation
def valid_coord(lat, lon):
    try:
        lat, lon = float(lat), float(lon)
        return 6 <= lat <= 37.5 and 68 <= lon <= 97.5 and lat != 0 and lon != 0
    except: return False

df['has_valid_coords'] = df.apply(lambda r: valid_coord(r.get('latitude'), r.get('longitude')), axis=1)

assert len(df) == len(df_raw), 'Record count mismatch!'
print(f'Cleaned dataset: {len(df)} records, {len(df.columns)} columns')
print(f'All {len(df_raw)} records preserved: OK')

In [ ]:
# ── Cell 7: State distribution chart ─────────────────────────────────────────
state_counts = df['state'].value_counts().reset_index()
state_counts.columns = ['state', 'count']
state_counts = state_counts.sort_values('count', ascending=True)

fig = px.bar(
    state_counts, x='count', y='state', orientation='h',
    color='count', color_continuous_scale='Reds',
    title='Blood Banks by State (Indian Blood Bank Directory)',
    labels={'count': 'Blood Banks', 'state': 'State'},
    text='count',
)
fig.update_traces(textposition='outside')
fig.update_layout(height=max(500, len(state_counts) * 22), showlegend=False)
fig.show()

In [ ]:
# ── Cell 8: Category and service time ────────────────────────────────────────
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'bar'}]],
                    subplot_titles=['Category Breakdown', 'Service Time (Normalised)'])

cat_vc = df['category'].value_counts()
fig.add_trace(go.Pie(labels=cat_vc.index, values=cat_vc.values, hole=0.4,
                     name='Category'), row=1, col=1)

svc_vc = df['service_time'].value_counts()
fig.add_trace(go.Bar(x=svc_vc.index, y=svc_vc.values, name='Service Time',
                     marker_color=['#E74C3C','#F39C12','#95A5A6','#7F8C8D']),
              row=1, col=2)

fig.update_layout(title='Category and Service Time Distribution', height=450)
fig.show()

In [ ]:
# ── Cell 9: Map ───────────────────────────────────────────────────────────────
map_df = df[df['has_valid_coords']].copy()
print(f'Mapping {len(map_df)} blood banks with valid coordinates.')

fig = px.scatter_map(
    map_df, lat='latitude', lon='longitude',
    color='category',
    hover_name='bank_name',
    hover_data={'state': True, 'district': True, 'category': True},
    zoom=4, center={'lat': 20.5, 'lon': 78.9},
    height=600,
    map_style='open-street-map',
    title=f'Indian Blood Bank Locations ({len(map_df):,} mapped)',
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.show()

In [ ]:
# ── Cell 10: Data quality visualization ──────────────────────────────────────
skip_cols = {'sr_no', 'has_valid_coords', 'service_time_raw'}
cols = [c for c in df.columns if c not in skip_cols]

quality_data = []
for col in cols:
    missing = df[col].isna().sum()
    if df[col].dtype == object:
        missing += (df[col] == '').sum()
    quality_data.append({
        'Column': col,
        '% Complete': round((len(df) - min(missing, len(df))) / len(df) * 100, 1)
    })

qdf = pd.DataFrame(quality_data).sort_values('% Complete')
colors = ['#E74C3C' if p < 50 else '#F39C12' if p < 80 else '#1ABC9C' for p in qdf['% Complete']]

fig = go.Figure(go.Bar(
    x=qdf['% Complete'], y=qdf['Column'], orientation='h',
    marker_color=colors, text=[f"{p:.0f}%" for p in qdf['% Complete']],
    textposition='outside',
))
fig.update_layout(
    title='Data Completeness by Column',
    xaxis_title='% Complete', xaxis_range=[0,115],
    height=max(500, len(qdf) * 22)
)
fig.show()

In [ ]:
# ── Cell 11: Summary stats ────────────────────────────────────────────────────
print('=== SUMMARY ===')
print(f'Total records    : {len(df):,}')
print(f'States           : {df["state"].nunique()}')
print(f'Districts        : {df["district"].nunique()}')
print(f'Valid coords     : {df["has_valid_coords"].sum():,} ({df["has_valid_coords"].mean()*100:.1f}%)')
print()
print('Category breakdown:')
print(df['category'].value_counts())
print()
print('Service time breakdown (normalised):')
print(df['service_time'].value_counts())
print()
print('Blood component available:')
print(df['blood_component'].value_counts())